# 01 -- Data + RAG index

First thing to run. Pulls MTS-Dialog (training) and ACI-Bench (held-out eval), then builds the FAISS retrieval index over the training set.

**Why Drive instead of Colab's local disk:** Colab wipes local files when the runtime disconnects. Cloning into Drive means `data/`, `outputs/`, and everything else this notebook produces survives across sessions -- you can close this tab, come back tomorrow, and pick up from the next notebook without re-running anything.

**If your GitHub repo is private, you need a token before running the cell below.**

1. On GitHub: Settings -> Developer settings -> Personal access tokens -> Fine-grained tokens -> Generate new token. Scope it to just this repo, permission `Contents: Read and write` (write lets you push results back from Colab later if you want to, e.g. in notebook 7).
2. In this Colab notebook: click the key icon (🔑) in the left sidebar -> Secrets -> Add new secret. Name it exactly `GITHUB_TOKEN`, paste the token as the value, and toggle notebook access on.

That keeps the token out of the notebook file itself -- pasting it directly into a cell would leak it into git history the moment this notebook gets committed. If your repo is public, skip this entirely, the cell below works without a token.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/MedAlignRL'
GITHUB_USERNAME = 'YOUR_USERNAME'   # <-- change this
GITHUB_REPO = 'MedAlignRL'         # <-- change if you named it differently

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
    print("No GITHUB_TOKEN secret found. Fine if your repo is public -- if it's "
          "private this clone will fail. See the setup note above.")

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
else:
    REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if not os.path.exists(PROJECT_DIR):
    print("Cloning into Drive (first time)...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already in Drive, pulling latest...")
    !cd {PROJECT_DIR} && git remote set-url origin {REPO_URL} && git fetch origin && git reset --hard origin/main

%cd {PROJECT_DIR}
!pip install -q -U -r requirements-colab.txt
!pip uninstall -y -q torchao  # Colab preinstalls an old torchao; peft raises ImportError on it during LoRA dispatch, and this repo never uses it


Quick GPU check -- worth knowing what you got before the longer runs later.

In [ ]:
!nvidia-smi

In [ ]:
!python -m spacy download en_core_web_sm -q

### Optional: scispaCy's clinical NER model
Better entity extraction than the general-English fallback. Skip if you're in a hurry, `reward.py` falls back automatically.

In [ ]:
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

### Build the splits and the index

In [ ]:
%cd {PROJECT_DIR}/src
!python data.py
!python rag.py --build-index

Done. `../data/` on Drive now has the train/val/test splits, the ACI-Bench eval set, and the FAISS index. Move on to `02_generate_preference_pairs.ipynb` whenever.